# MYRAA Kaggle GPU Worker (T4x2)

Ephemeral compute worker for MYRAA. This notebook:

1. validates the environment (deps, CUDA, FP16, multi-GPU, NCCL)
2. prints a capability report (paste it into MYRAA as a worker heartbeat)
3. runs one MYRAA job (paste the exported job JSON or drop it in `/kaggle/input/`)
4. writes a result JSON (import it back into MYRAA)

Set `MODEL_ID` below or let the job specify it. Never paste secrets into this notebook.

In [ ]:
# ---- MYRAA worker configuration ----
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # default smoke-test model (~1.1B, fits one T4)
QUANTIZATION = "4bit"   # 4bit | 8bit | none
DTYPE = "fp16"          # fp16 (T4 default) | fp32 ; T4 has no fast bf16
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.7
JOB_JSON_INLINE = ""      # optional: paste a MYRAA exported job JSON here
JOB_FILE = "/kaggle/input/myraa_job.json"  # or drop the file here
RESULT_PATH = "/kaggle/working/myraa_result.json"
CAPS_PATH = "/kaggle/working/myraa_caps.json"
print("config ok:", MODEL_ID, QUANTIZATION, DTYPE)

In [ ]:
import importlib, sys
REQUIRED = ["torch", "transformers", "accelerate", "bitsandbytes", "safetensors"]
missing = []
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing.append(pkg)
print("torch:", importlib.import_module("torch").__version__ if "torch" not in missing else "MISSING")
print("missing:", missing if missing else "none")
if missing:
    raise SystemExit("Install missing packages first: pip install " + " ".join(missing))

In [ ]:
import json, subprocess, torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
n = torch.cuda.device_count()
print("GPU count:", n)
gpus = []
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    vram = round(p.total_memory / 1024**3, 2)
    gpus.append({"index": i, "model": p.name, "vram_gb": vram,
                  "compute_capability": f"{p.major}.{p.minor}",
                  "multi_processor_count": p.multi_processor_count})
    print(f"GPU {i}: {p.name}, {vram} GB, cc {p.major}.{p.minor}")
agg = round(sum(g["vram_gb"] for g in gpus), 2)
single = max([g["vram_gb"] for g in gpus], default=0)
print("single_gpu_vram_gb:", single, " aggregate_vram_gb:", agg)
assert n >= 1, "no CUDA GPUs visible"
CAPS = {"provider": "kaggle", "gpu_count": n, "gpus": gpus,
        "single_gpu_vram_gb": single, "aggregate_vram_gb": agg,
        "cuda_version": torch.version.cuda, "pytorch_version": torch.__version__}

In [ ]:
import time, torch
USE_FP16 = DTYPE == "fp16"
# 1) CUDA alloc
x = torch.randn(1024, 1024, device="cuda:0")
print("cuda alloc ok")
# 2) FP16 matmul on GPU 0
if USE_FP16:
    a = torch.randn(2048, 2048, device="cuda:0", dtype=torch.float16)
    b = torch.randn(2048, 2048, device="cuda:0", dtype=torch.float16)
    torch.cuda.synchronize()
    t0 = time.time()
    c = a @ b
    torch.cuda.synchronize()
    print(f"fp16 matmul ok ({round(time.time()-t0, 3)}s)")
# 3) simultaneous multi-GPU
multi_ok = False
if torch.cuda.device_count() >= 2:
    a0 = torch.randn(4096, 4096, device="cuda:0")
    b0 = torch.randn(4096, 4096, device="cuda:0")
    a1 = torch.randn(4096, 4096, device="cuda:1")
    b1 = torch.randn(4096, 4096, device="cuda:1")
    torch.cuda.synchronize()
    t0 = time.time()
    x0 = a0 @ b0
    x1 = a1 @ b1
    torch.cuda.synchronize()
    multi_ok = True
    print(f"multi-gpu ok ({round(time.time()-t0, 3)}s)")
# 4) NCCL presence
try:
    import torch.distributed as dist
    nccl = dist.is_nccl_available()
except Exception:
    nccl = False
print("nccl available:", nccl)
CAPS.update({"fp16": USE_FP16, "bf16": False, "nccl": bool(nccl),
               "multi_gpu": multi_ok or torch.cuda.device_count() >= 2})
with open(CAPS_PATH, "w") as f:
    json.dump(CAPS, f, indent=2)
print("capability report:", CAPS_PATH)
print(json.dumps(CAPS, indent=2))

In [ ]:
import json, os, time, torch

job = None
if JOB_JSON_INLINE.strip():
    job = json.loads(JOB_JSON_INLINE)
elif os.path.exists(JOB_FILE):
    with open(JOB_FILE) as f:
        job = json.load(f)
if job is None:
    job = {"job_id": "manual-" + str(int(time.time())),
             "job_type": "LLM_INFERENCE", "model": {"id": MODEL_ID},
             "input": {"prompt": "Explain in one sentence why GPUs help neural networks."}}
    print("no job supplied - running default smoke prompt")
model_id = (job.get("model") or {}).get("id") or MODEL_ID
need = job.get("max_vram_gb")
single = CAPS["single_gpu_vram_gb"]
if need and single and need > single:
    sup = ((job.get("model") or {}).get("supportsSharding")) is True
    if not sup:
        raise SystemExit(f"MODEL_VRAM_INSUFFICIENT: job needs {need}GB on one GPU, single capacity {single}GB, no sharding")
    print(f"sharded placement: {need}GB across GPUs (aggregate {CAPS['aggregate_vram_gb']}GB)")
print("job:", job.get("job_id"), job.get("job_type"), "model:", model_id)

In [ ]:
import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

t0 = time.time()
tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
qcfg = None
if QUANTIZATION == "4bit":
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16 if DTYPE == "fp16" else torch.float32)
elif QUANTIZATION == "8bit":
    qcfg = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=qcfg,
    torch_dtype=torch.float16 if DTYPE == "fp16" else torch.float32,
    device_map="auto", trust_remote_code=False)
load_s = round(time.time() - t0, 1)
mem0 = round(torch.cuda.memory_allocated(0) / 1024**3, 2)
print(f"model loaded in {load_s}s, gpu0 allocated {mem0}GB")

prompt_in = job.get("input") or {}
prompt = prompt_in.get("prompt") or prompt_in.get("text") or "Say hello."
inputs = tok(prompt, return_tensors="pt").to(model.device)
t1 = time.time()
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                          temperature=TEMPERATURE, do_sample=TEMPERATURE > 0,
                          pad_token_id=tok.eos_token_id)
gen_s = round(time.time() - t1, 2)
text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
in_tok = int(inputs["input_ids"].numel())
out_tok = int(out[0].numel() - in_tok)
tps = round(out_tok / max(gen_s, 0.01), 1)
mem1 = round(torch.cuda.memory_allocated(0) / 1024**3, 2)
print(f"generated {out_tok} tokens in {gen_s}s ({tps} tok/s), peak gpu0 {mem1}GB")
print(text[:2000])

result = {"job_id": job.get("job_id"), "status": "COMPLETED", "result": {"text": text},
          "metrics": {"input_tokens": in_tok, "output_tokens": out_tok, "tokens_per_sec": tps,
                      "load_seconds": load_s, "gen_seconds": gen_s, "gpu0_gb": mem1,
                      "model": model_id, "quantization": QUANTIZATION, "dtype": DTYPE}}
with open(RESULT_PATH, "w") as f:
    json.dump(result, f, indent=2)
print("result artifact:", RESULT_PATH)
print("Paste the capability JSON into MYRAA heartbeat, and import the result JSON via MYRAA Compute Center.")